In [7]:
# Q1

import time
import torchvision
from torchvision import transforms
from d2l import torch as d2l

class FashionMNIST(d2l.DataModule):
    def __init__(self, batch_size=64, resize=(28, 28)):
        super().__init__()
        self.save_hyperparameters()
        trans = transforms.Compose([transforms.Resize(resize),
                                    transforms.ToTensor()])
        self.train = torchvision.datasets.FashionMNIST(
            root=self.root, train=True, transform=trans, download=True)
        self.val = torchvision.datasets.FashionMNIST(
            root=self.root, train=False, transform=trans, download=True)

    def get_dataloader(self, train):
        data = self.train if train else self.val
        return __import__('torch').utils.data.DataLoader(
            data, self.batch_size, shuffle=train,
            num_workers=self.num_workers)

In [8]:
batch_sizes = [1, 8, 64, 256]
results = {}

for bs in batch_sizes:
    d = FashionMNIST(batch_size=bs, resize=(32, 32))
    tic = time.time()
    for X, y in d.train_dataloader():
        continue
    elapsed = time.time() - tic
    results[bs] = elapsed
    print(f'batch_size={bs:>4d}  batches={len(d.train)//bs:>5d}  time={elapsed:.2f}s')

batch_size=   1  batches=60000  time=33.43s
batch_size=   8  batches= 7500  time=14.33s
batch_size=  64  batches=  937  time=12.09s
batch_size= 256  batches=  234  time=11.65s


In [9]:
# Q2
import torch, cProfile, pstats

d_base = FashionMNIST(batch_size=64, resize=(32, 32))
pr = cProfile.Profile()
pr.enable()
for X, y in d_base.train_dataloader(): continue
pr.disable()

stats = pstats.Stats(pr).sort_stats('cumulative')
print(f"{'ncalls':>8} {'tottime':>9} {'cumtime':>9}  location")
for i, ((file, lineno, func), (cc, nc, tt, ct, _)) in enumerate(stats.stats.items()):
    if i >= 8: break
    print(f"{nc:>8} {tt:>9.3f} {ct:>9.3f}  {file.split('/')[-1].split(chr(92))[-1]}:{lineno}({func})")

configs = {
    'baseline (num_workers=0)': dict(num_workers=0, pin_memory=False, persistent_workers=False),
    'num_workers=4':            dict(num_workers=4, pin_memory=False, persistent_workers=False),
    '+ pin_memory':             dict(num_workers=4, pin_memory=True,  persistent_workers=False),
    '+ persistent_workers':     dict(num_workers=4, pin_memory=True,  persistent_workers=True),
}
print("\n--- optimization comparison ---")
for name, kw in configs.items():
    loader = torch.utils.data.DataLoader(d_base.train, batch_size=64, shuffle=True, **kw)
    tic = time.time()
    for X, y in loader: continue
    print(f'{name:<30s} {time.time()-tic:.2f}s')

  ncalls   tottime   cumtime  location
     939     0.005     0.007  profiler.py:787(__init__)
     939     0.004     0.062  profiler.py:800(__enter__)
     939     0.010     0.087  profiler.py:806(__exit__)
     939     0.000     0.000  __init__.py:129(annotate)
       8     0.000     0.000  __init__.py:77(get_sharing_strategy)
    1884     0.005     0.006  reductions.py:32(__init__)
       8     0.000     0.000  util.py:44(sub_debug)
    2064     0.001     0.002  reductions.py:45(expired)

--- optimization comparison ---
baseline (num_workers=0)       4.78s
num_workers=4                  8.83s
+ pin_memory                   8.78s
+ persistent_workers           7.88s
